In [1]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import cartopy.crs as ccrs
import cartopy.feature as cfeature


# --- configuration (match Fig5/01_peak_snapshots.ipynb quiver settings) ---
CASES = [
    {
        "name": "Ragasa",
        "input_dir": "ragasa",
        "time": "20250923 2200",
        "station_sheet": "Ragasa",
    },
    {
        "name": "Yagi",
        "input_dir": "yagi",
        "time": "20240905 1200",
        "station_sheet": "Yagi",
    },
]

QUIVER_SCALE = 500
QUIVER_WIDTH = 0.0050
QUIVER_HEADWIDTH = 3.0
QUIVER_COLOR = "red"

FIGSIZE = (6, 6)
DPI = 600
MAP_PADDING = 0.08  # degrees

# in-figure quiver notation
QUIVERKEY_U = 30
QUIVERKEY_LABEL = rf"{QUIVERKEY_U} m s$^{{-1}}$"
QUIVERKEY_FONTSIZE = 18
QUIVERKEY_X = 0.08
QUIVERKEY_Y = 0.93

plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 18


def _find_fig5_dirs() -> tuple[Path, Path]:
    """Return the Fig5 input and output directories from common CWDs."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "Fig5", cwd / "Figs" / "Fig5"]
    candidates.extend(parent / "Fig5" for parent in cwd.parents)

    for fig5_dir in candidates:
        input_dir = fig5_dir / "input"
        if (input_dir / "weather_station_location.xlsx").exists():
            output_dir = fig5_dir / "output"
            output_dir.mkdir(parents=True, exist_ok=True)
            return input_dir, output_dir

    raise FileNotFoundError(
        "Could not find Fig5/input/weather_station_location.xlsx "
        f"from current working directory: {cwd}"
    )


def _normalise_station_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalise CSV station headers while preserving timestamp_utc."""
    rename = {
        column: column.strip().lower()
        for column in df.columns
        if column != "timestamp_utc"
    }
    normalised = list(rename.values())
    if len(normalised) != len(set(normalised)):
        raise ValueError("Duplicate station columns after normalising CSV headers")
    return df.rename(columns=rename)


def _load_target_wind(
    wspd_path: Path,
    wdir_path: Path,
    target_time: str,
) -> tuple[pd.Series, pd.Series, list[str]]:
    """Load one exact observation time and validate station consistency."""
    df_wspd = pd.read_csv(wspd_path, dtype={"timestamp_utc": str})
    df_wdir = pd.read_csv(wdir_path, dtype={"timestamp_utc": str})

    for path, df in ((wspd_path, df_wspd), (wdir_path, df_wdir)):
        if "timestamp_utc" not in df.columns:
            raise ValueError(f"Missing timestamp_utc column in {path}")
        df["timestamp_utc"] = df["timestamp_utc"].astype(str).str.strip()

    df_wspd = _normalise_station_columns(df_wspd)
    df_wdir = _normalise_station_columns(df_wdir)

    wspd_match = df_wspd[df_wspd["timestamp_utc"] == target_time]
    wdir_match = df_wdir[df_wdir["timestamp_utc"] == target_time]
    if len(wspd_match) != 1 or len(wdir_match) != 1:
        raise ValueError(
            f"Expected exactly one speed and direction row at {target_time}; "
            f"found {len(wspd_match)} and {len(wdir_match)}"
        )

    wspd_stations = [c for c in df_wspd.columns if c != "timestamp_utc"]
    wdir_stations = [c for c in df_wdir.columns if c != "timestamp_utc"]
    if set(wspd_stations) != set(wdir_stations):
        only_speed = sorted(set(wspd_stations) - set(wdir_stations))
        only_direction = sorted(set(wdir_stations) - set(wspd_stations))
        raise ValueError(
            "Wind speed/direction station columns differ: "
            f"speed only={only_speed}, direction only={only_direction}"
        )

    wspd_row = wspd_match.iloc[0].drop(labels="timestamp_utc")
    wdir_row = wdir_match.iloc[0].drop(labels="timestamp_utc")
    return wspd_row, wdir_row, wspd_stations


def _load_station_locations(
    xlsx_path: Path,
    sheet_name: str,
    station_order: list[str],
) -> pd.DataFrame:
    """Load a headerless station/latitude/longitude worksheet."""
    loc = pd.read_excel(
        xlsx_path,
        sheet_name=sheet_name,
        header=None,
        usecols=[0, 1, 2],
        names=["station", "lat", "lon"],
    )
    loc["station"] = loc["station"].astype(str).str.strip().str.lower()
    loc["lat"] = pd.to_numeric(loc["lat"], errors="coerce")
    loc["lon"] = pd.to_numeric(loc["lon"], errors="coerce")
    loc = loc.dropna(subset=["station", "lat", "lon"])

    duplicated = sorted(loc.loc[loc["station"].duplicated(), "station"].unique())
    if duplicated:
        raise ValueError(f"Duplicate station coordinates in sheet {sheet_name!r}: {duplicated}")

    loc = loc.set_index("station")
    missing = sorted(set(station_order) - set(loc.index))
    if missing:
        raise ValueError(f"Stations missing coordinates in sheet {sheet_name!r}: {missing}")

    return loc.loc[station_order].reset_index()


def _winddir_to_uv(
    wspd: np.ndarray,
    wdir_deg_from: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Convert speed and meteorological direction-from to u/v in m/s."""
    theta = np.deg2rad(wdir_deg_from)
    u = -wspd * np.sin(theta)
    v = -wspd * np.cos(theta)
    return u, v


def _map_extent(loc: pd.DataFrame) -> list[float]:
    return [
        float(loc["lon"].min()) - MAP_PADDING,
        float(loc["lon"].max()) + MAP_PADDING,
        float(loc["lat"].min()) - MAP_PADDING,
        float(loc["lat"].max()) + MAP_PADDING,
    ]


def _plot_station_quiver(
    loc: pd.DataFrame,
    wspd_row: pd.Series,
    wdir_row: pd.Series,
    out_path: Path,
) -> None:
    station_order = loc["station"].to_numpy()
    wspd = pd.to_numeric(wspd_row.reindex(station_order), errors="coerce").to_numpy(dtype=float)
    wdir = pd.to_numeric(wdir_row.reindex(station_order), errors="coerce").to_numpy(dtype=float)

    # Treat 0-speed + 0-direction as missing (common encoding).
    missing = (np.isclose(wspd, 0.0) & np.isclose(wdir, 0.0)) | np.isnan(wspd) | np.isnan(wdir)
    u, v = _winddir_to_uv(wspd, wdir)
    u = u.astype(float)
    v = v.astype(float)
    u[missing] = np.nan
    v[missing] = np.nan

    fig = plt.figure(figsize=FIGSIZE)
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(_map_extent(loc), crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN.with_scale("10m"))
    ax.add_feature(cfeature.LAND.with_scale("10m"))

    q = ax.quiver(
        loc["lon"].to_numpy(),
        loc["lat"].to_numpy(),
        u,
        v,
        color=QUIVER_COLOR,
        scale=QUIVER_SCALE,
        width=QUIVER_WIDTH,
        headwidth=QUIVER_HEADWIDTH,
        transform=ccrs.PlateCarree(),
    )
    ax.quiverkey(
        q,
        X=QUIVERKEY_X,
        Y=QUIVERKEY_Y,
        U=QUIVERKEY_U,
        label=QUIVERKEY_LABEL,
        labelpos="E",
        coordinates="axes",
        fontproperties={"family": "Arial", "size": QUIVERKEY_FONTSIZE},
    )

    plt.savefig(out_path, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


input_dir, output_dir = _find_fig5_dirs()
station_xlsx = input_dir / "weather_station_location.xlsx"
print(f"Using input directory: {input_dir}")
print(f"Using output directory: {output_dir}")
print(f"Using station coordinates: {station_xlsx}")

for case in CASES:
    case_dir = input_dir / case["input_dir"]
    wspd_row, wdir_row, station_order = _load_target_wind(
        case_dir / "mean_wspd_obs.csv",
        case_dir / "mean_wdir_obs.csv",
        case["time"],
    )
    loc = _load_station_locations(
        station_xlsx,
        case["station_sheet"],
        station_order,
    )
    out_path = output_dir / f"wind_obs_stations_{case['time'].replace(' ', '_')}.tif"
    _plot_station_quiver(loc, wspd_row, wdir_row, out_path)
    print(
        f"Saved {case['name']} at {case['time']} using {len(loc)} stations: "
        f"{out_path}"
    )

print("Done.")


Using input directory: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Fig5\input
Using output directory: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Fig5\output
Using station coordinates: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Fig5\input\weather_station_location.xlsx


Saved Ragasa at 20250923 2200 using 30 stations: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Fig5\output\wind_obs_stations_20250923_2200.tif


Saved Yagi at 20240905 1200 using 31 stations: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Fig5\output\wind_obs_stations_20240905_1200.tif
Done.
